# **Bi-GRU** for Temporal Behavior Learning

In [ ]:
# removing warnings
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import gc
import os
import tensorflow as tf

from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.models import Sequential
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.layers import Bidirectional, GRU, Dense

import matplotlib as plt
from sklearn.model_selection import KFold
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import confusion_matrix, classification_report

In [ ]:
# no need to write "0"
os.environ["CUDA_VISIBLE_DEVICES"]="1,2,3,4"
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2' 

## Loading Data

In [ ]:
BASE_PATH = "<path to experiment directory>"
MODEL_DATA_PATH = f"{BASE_PATH}/<path to load KAN-AE encoded data>"
MODEL_PATH = f"{BASE_PATH}/<path to export Bi-GRU model>"

In [ ]:
## encoded data as input (comes from KANAE+PCA)
X_train = np.load(f'{MODEL_DATA_PATH}X_train_pca_enc.npy')
X_val = np.load(f'{MODEL_DATA_PATH}X_val_pca_enc.npy')
X_test = np.load(f'{MODEL_DATA_PATH}X_test_pca_enc.npy')

y_train = np.load(f'{MODEL_DATA_PATH}y_train.npy')
y_val = np.load(f'{MODEL_DATA_PATH}y_val.npy')
y_test = np.load(f'{MODEL_DATA_PATH}y_test.npy')

In [ ]:
X_train.shape, X_val.shape, X_test.shape

# Expected:   (N, seq, features), each

## Model Building

In [ ]:
CLASS_COUNTS = len(set(y_train))
CLASS_COUNTS

In [ ]:
input_shape = X_train.shape[1:]
input_shape

---

### Model Architecture

In [ ]:
def get_bigru(input_dim, paramTuple):
    unit_l1, drop_l1, rec_drop_l1, unit_l2, drop_l2, rec_drop_l2, unit_l3, drop_l3, rec_drop_l3 = paramTuple
    
    model = tf.keras.Sequential()

    model.add(Bidirectional(
        GRU(unit_l1,
            activation="tanh",
            return_sequences=True,
            dropout=drop_l1, # additional suggestions
            recurrent_dropout=rec_drop_l1),  # strict suggestions
        input_shape=input_dim
    )) # additional suggestions
    model.add(tf.keras.layers.LayerNormalization()) # strict suggestions

    model.add(Bidirectional(
        GRU(unit_l2,
            activation="tanh",
            return_sequences=True,
            dropout=drop_l2, # additional suggestions
            recurrent_dropout=rec_drop_l2) # strict suggestions
    )) # additional suggestions
    model.add(tf.keras.layers.LayerNormalization()) # strict suggestions

    model.add(Bidirectional(
        GRU(unit_l3,
            activation="tanh",
            return_sequences=False,
            dropout=drop_l3, # additional suggestions
            recurrent_dropout=rec_drop_l3) # strict suggestions
    ))

    model.add(Dense(CLASS_COUNTS, activation="softmax"))

    return model

In [ ]:
# training param set
unit_l1 = 432
unit_l2 = 112
unit_l3 = 96
drop_l1 = 0.2
drop_l2 = 0.3
drop_l3 = 0.1
rec_drop_l1 = 0.2
rec_drop_l2 = 0.2
rec_drop_l3 = 0.0

learning_rate =  9.07e-4
batch_size = 16 # tune as per the GPU (if available)

allParamList = (unit_l1, drop_l1, rec_drop_l1, 
             unit_l2, drop_l2, rec_drop_l2, 
             unit_l3, drop_l3, rec_drop_l3)

## Training with Cross Validation

In [ ]:
# Define the number of folds
n_folds = 3
# other param
EPOCH_COUNT=30 
BATCH_SIZE=batch_size
LR=learning_rate 

# Define empty lists to store results
acc_per_fold = []
loss_per_fold = []

<font color='red'> **X_val and y_val are not used explicitely in k-fold cross validation** </font>

In [ ]:
# merging (val + train)
X_train_joined = np.concatenate([X_train, X_val], axis=0)
y_train_joined = np.concatenate([y_train, y_val], axis=0)

In [ ]:
X_train_joined.shape, X_test.shape

---

In [ ]:
## before training; Multi-GPU strategy
strategy = tf.distribute.MirroredStrategy()
print("Number of GPUs:", strategy.num_replicas_in_sync)

In [ ]:
GLOBAL_BATCH_SIZE = BATCH_SIZE * strategy.num_replicas_in_sync

# KFold cross validation
kfold = KFold(n_splits=n_folds, shuffle=True, random_state=1)

In [ ]:
# KFold Cross Validation
for fold, (train, val) in enumerate(
        kfold.split(X_train_joined, y_train_joined),
        start=1):

    print(f"\nFold {fold}")

    # ---------------------------------------------------------
    # Cleanup from previous fold
    # ---------------------------------------------------------
    tf.keras.backend.clear_session()
    gc.collect()

    # ---------------------------------------------------------
    # Split current fold data
    # ---------------------------------------------------------
    X_tr = X_train_joined[train]
    y_tr = y_train_joined[train]

    X_val = X_train_joined[val]
    y_val = y_train_joined[val]

    print("Training X shape :", X_tr.shape)
    print("Training y shape :", y_tr.shape)

    # ---------------------------------------------------------
    # Create TensorFlow datasets
    # ---------------------------------------------------------

    # Training dataset
    train_ds = tf.data.Dataset.from_tensor_slices((X_tr, y_tr))

    # Shuffle training data
    train_ds = train_ds.shuffle(buffer_size=len(X_tr))

    # Batch data for multi-GPU training
    # GLOBAL_BATCH_SIZE = BATCH_SIZE * number_of_GPUs
    train_ds = train_ds.batch(
        GLOBAL_BATCH_SIZE,
        drop_remainder=True
    )

    # Conservative prefetching for notebook stability
    train_ds = train_ds.prefetch(1)

    # Validation dataset
    val_ds = tf.data.Dataset.from_tensor_slices((X_val, y_val))

    val_ds = val_ds.batch(
        GLOBAL_BATCH_SIZE,
        drop_remainder=False
    )

    val_ds = val_ds.prefetch(1)
    
    #### deterministic dataset options to improve MirroredStrategy 
    #### stability and avoid kernal crash/restart
    options = tf.data.Options()
    options.experimental_distribute.auto_shard_policy = (
        tf.data.experimental.AutoShardPolicy.DATA
    )
    train_ds = train_ds.with_options(options)
    val_ds = val_ds.with_options(options)

    # ---------------------------------------------------------
    # Early stopping
    # ---------------------------------------------------------
    stop_early = tf.keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=3,
        restore_best_weights=True
    )

    # ---------------------------------------------------------
    # Create model inside strategy scope (MULTI-GPU)
    # ---------------------------------------------------------
    with strategy.scope():

        model_bigru = get_bigru(input_shape, allParamList)

        model_bigru.compile(
            loss='sparse_categorical_crossentropy',
            optimizer=Adam(learning_rate=LR),
            metrics=['accuracy']
        )

    # ---------------------------------------------------------
    # Train model
    # ---------------------------------------------------------
    history = model_bigru.fit(
        train_ds,
        validation_data=val_ds,
        epochs=EPOCH_COUNT,
        callbacks=[stop_early],
        verbose=1
    )

    # ---------------------------------------------------------
    # Evaluate model
    # ---------------------------------------------------------
    loss, acc = model_bigru.evaluate(
        val_ds,
        verbose=0
    )

    print(f"Fold {fold} - Loss: {loss}, Accuracy: {acc}")

    acc_per_fold.append(acc)
    loss_per_fold.append(loss)

    # ---------------------------------------------------------
    # Cleanup after fold (for notebook stability)
    # ---------------------------------------------------------
    del train_ds
    del val_ds

    gc.collect()

### Evaluate and Export

In [ ]:
# final validation results 
print(f"Average Loss across {n_folds}-folds:", np.mean(loss_per_fold))
print(f"Average Accuracy across {n_folds}-folds:", np.mean(acc_per_fold))

In [ ]:
# testing on genuine avatars (test data)
loss, accuracy = model_bigru.evaluate(X_test, y_test)
print("Test accuracy:", accuracy)

In [ ]:
## exporting
model_bigru.save(f"{MODEL_PATH}/bigru_classifier.keras")

**`.keras` models are more comprehensive than `.h5` and are also human readable**

---  
--- 